# Field API: GSI DEM to spatial spectrum

Set `GWEXPY_DEM_TILES` to an os.pathsep-separated list of your own GSI FGD DEM GML/ZIP files. Without files, this notebook runs on a synthetic geographic raster. It demonstrates read, explicit missing-value filling, local Cartesian conversion, and spatial FFT.

## Setup

In [ ]:
import os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from gwexpy.fields import GeoMetadata, ScalarField

## Read or synthesize terrain

In [ ]:
paths = [Path(p) for p in os.environ.get("GWEXPY_DEM_TILES", "").split(os.pathsep) if p]
if paths:
    missing = [str(p) for p in paths if not p.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    dem = ScalarField.read(paths, "elevation", format="gsi-dem")
else:
    # Offline geographic raster; rows descend from north to south.
    lon = np.linspace(137.30, 137.32, 64) * u.deg
    lat = np.linspace(36.42, 36.40, 48) * u.deg
    xx, yy = np.meshgrid(lon.value, lat.value, indexing="ij")
    elevation = 400 + 180 * np.exp(-((xx - 137.31) / 0.004)**2 - ((yy - 36.41) / 0.005)**2)
    elevation[20:24, 15:19] = np.nan
    dem = ScalarField(elevation[None, :, :, None], unit=u.m, name="elevation",
        axis0=[0.0] * u.s, axis1=lon, axis2=lat, axis3=[0.0] * u.m,
        axis_names=("t", "lon", "lat", "z"),
        space_domain={"lon": "angle", "lat": "angle", "z": "real"})
    dem.geo = GeoMetadata(crs="EPSG:6668", source_format="synthetic", active=True)
print(dem.shape, dem.axis_names, dem.geo)

## Fill, project, and transform

In [ ]:
# NoData remains NaN after read; infilling is explicit.
filled = dem.fill_missing(method="nearest")
assert np.isfinite(filled.value).all()
local = filled.to_local_cartesian()
assert local.space_domains["x"] == "real"
# Forward scipy.fft.fftn is unnormalized; the output axes are angular wavenumber.
spectrum = local.fft_space(["x", "y"], spacing_rtol=1e-8)
power = np.abs(spectrum.value[0, :, :, 0])**2
print(spectrum.shape, spectrum.axis_names, spectrum.geo.history[-1])

## Inspect terrain and spectrum

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(local.value[0, :, :, 0].T, origin="lower", aspect="auto")
axes[0].set_title("Elevation in local frame [m]")
axes[1].imshow(np.log1p(np.fft.fftshift(power)).T, origin="lower", aspect="auto")
axes[1].set_title("log(1 + spatial spectral power)")
for ax in axes:
    ax.set_xlabel("grid x index")
    ax.set_ylabel("grid y index")
fig.tight_layout()
plt.show()

The local tangent-plane conversion is an approximation for a small area. For wide areas use `to_projected()` with the optional `geospatial` extra. `spacing_rtol` explicitly permits approximately uniform samples. The forward FFT is unnormalized; its coordinates are angular wavenumbers.